# Week 2 Assignment: SARSA for Dynamic Pricing in Retail/E-Commerce

**Goal:** Use tabular SARSA (on-policy RL) to learn an optimal **sequential pricing policy** that maximizes revenue over a sales period (e.g., 10 weeks) for a product, given fluctuating demand.

In data science, dynamic pricing uses historical sales data to adjust prices adaptively. Here, we treat pricing as an MDP:
- **States:** Time remaining (early/mid/late) + current demand level (low/medium/high)
- **Actions:** Set price tier (Low, Medium, High)
- **Rewards:** Revenue = units sold × price; units sold depend on price and demand (higher price → fewer sales, but more revenue per unit)

We'll derive demand sensitivity from data, then train SARSA to find the best policy.

**Dataset:** We use a small synthetic dataset mimicking Walmart-style weekly sales (inspired by Kaggle Walmart Recruiting Store Sales Forecasting). It includes weekly_sales, markdown effects, etc.

**Submission:** Complete all TODO sections, run the notebook, include plots and answers to questions.

In [1]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
from IPython.display import clear_output

# Set random seed for reproducibility
np.random.seed(42)

## Part 1: Data Loading & Exploration (Data Science Step)

We create a synthetic dataset of weekly sales for one product/store. In real life, you'd load from Kaggle `train.csv` + `features.csv`.

**TODO 1.1:** Run the cell below to generate the synthetic data. Then add 2–3 lines of pandas code to compute:
- Average weekly sales overall
- Average sales when IsHoliday == True vs False
- Quantiles for demand levels (0.33 and 0.66) on Weekly_Sales

In [2]:
# Synthetic data mimicking Walmart weekly sales (10 weeks × many episodes, but aggregated)
n_weeks = 10
n_samples = 200  # simulated historical weeks

data = pd.DataFrame({
    'Week': np.tile(np.arange(1, n_weeks+1), n_samples // n_weeks + 1)[:n_samples],
    'IsHoliday': np.random.choice([False, True], size=n_samples, p=[0.8, 0.2]),
    'MarkdownEffect': np.random.uniform(0, 0.4, n_samples),  # fraction sales boost from discount
    'Temperature': np.random.normal(60, 15, n_samples),
    'Weekly_Sales': np.random.lognormal(mean=8, sigma=1.2, size=n_samples) * (1 + np.random.uniform(-0.3, 0.5, n_samples))
})

# Make sales sensitive to 'demand' proxies
data['Weekly_Sales'] = data['Weekly_Sales'].clip(500, 8000)

print(data.head())

# TODO 1.1: Compute and print these statistics
avg_sales = _____  # overall mean Weekly_Sales
avg_holiday = _____  # mean when IsHoliday == True
avg_non_holiday = _____  # mean when False
demand_low, demand_high = np.quantile(data['Weekly_Sales'], [0.33, 0.66])

print(f"Average sales: {avg_sales:.0f}")
print(f"Holiday vs non-holiday: {avg_holiday:.0f} vs {avg_non_holiday:.0f}")
print(f"Demand quantiles: low < {demand_low:.0f}, high > {demand_high:.0f}")

   Week  IsHoliday  MarkdownEffect  Temperature  Weekly_Sales
0     1      False        0.256813    79.582182   2081.006753
1     2       True        0.033656    60.315058   8000.000000
2     3      False        0.064651    70.229295   7697.751939
3     4      False        0.359422    55.345999   1235.853251
4     5      False        0.242572    64.862495   3493.235630


NameError: name '_____' is not defined

## Part 2: Define MDP Components from Data

**States:**
- Time remaining: 3 bins (Early: weeks 8–10 left, Mid: 4–7, Late: 1–3) → index 0,1,2
- Demand level: 3 bins (Low, Medium, High) based on quantiles above → index 0,1,2
→ Total states: 3 × 3 = 9

**Actions:** 0=Low price (high volume, low margin), 1=Medium, 2=High price (low volume, high margin)

**TODO 2.1:** Define a function that, given current demand level and action, returns expected units sold (normalize sales to units assuming base price).

In [ ]:
# Price tiers (arbitrary units, e.g., $ per unit)
prices = [8.0, 10.0, 12.0]  # Low, Medium, High

# Demand sensitivity: how many 'units' sold at each price/demand
# (derived roughly from data: higher demand → more sales; higher price → fewer)
def expected_units(demand_level, action):
    # demand_level: 0=low, 1=med, 2=high
    base_units = [40, 80, 140]  # low/med/high demand at medium price
    price_factor = [1.4, 1.0, 0.7]  # low price → 40% more units, high → 30% less
    
    # TODO 2.1: Fill in the return statement
    return _____  # base_units[demand_level] * price_factor[action]

## Part 3: Custom Gym Environment

**TODO 3.1:** Complete the `step` function. Reward = units_sold × price - small holding cost if not sold out.

In [ ]:
class DynamicPricingEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Discrete(9)  # 3 time × 3 demand
        self.action_space = spaces.Discrete(3)       # price tiers
        self.max_steps = 10
        self.reset()
    
    def reset(self, seed=None, options=None):
        self.time_bin = 2  # start with 'Early' (high time left)
        # Sample demand level from historical distribution
        self.demand_level = np.random.choice([0,1,2], p=[0.33, 0.34, 0.33])
        self.steps = 0
        state = self._get_state()
        return state, {}
    
    def _get_state(self):
        return self.time_bin * 3 + self.demand_level
    
    def step(self, action):
        self.steps += 1
        units = expected_units(self.demand_level, action)
        revenue = units * prices[action]
        
        # Small cost if low sales (holding)
        reward = revenue - 50 if units < 30 else revenue
        
        # Transition: demand may change slightly, time decreases
        self.time_bin = max(0, self.time_bin - 1)
        if np.random.rand() < 0.25:
            self.demand_level = np.clip(self.demand_level + np.random.choice([-1,0,1]), 0, 2)
        
        done = (self.time_bin == 0) or (self.steps >= self.max_steps)
        truncated = False
        
        # TODO 3.1: If done and time_bin==0, add bonus/penalty if revenue high/low? (optional)
        if done and self.time_bin == 0:
            reward += _____  # e.g. +200 if revenue > threshold, else -100
        
        return self._get_state(), reward, done, truncated, {"revenue": revenue}

## Part 4: SARSA Implementation

**TODO 4.1:** Fill in the epsilon-greedy action selection.
**TODO 4.2:** Fill in the SARSA update rule.

In [ ]:
env = DynamicPricingEnv()
n_states = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))

# Hyperparameters
alpha = 0.15
gamma = 0.98
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
n_episodes = 8000

episode_rewards = []

for ep in range(n_episodes):
    state, _ = env.reset()
    action = _____  # TODO 4.1: epsilon-greedy
    # if np.random.rand() < epsilon: env.action_space.sample() else: np.argmax(Q[state])
    
    total_reward = 0
    done = False
    
    while not done:
        next_state, reward, done, truncated, _ = env.step(action)
        done = done or truncated
        
        next_action = _____  # TODO 4.1 again: epsilon-greedy on next_state
        
        # TODO 4.2: SARSA update
        td_target = reward + (0 if done else gamma * Q[next_state, next_action])
        td_error = td_target - Q[state, action]
        Q[state, action] += _____  # alpha * td_error
        
        state = next_state
        action = next_action
        total_reward += reward
    
    episode_rewards.append(total_reward)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

# Plot learning curve (moving average)
plt.figure(figsize=(10,5))
plt.plot(pd.Series(episode_rewards).rolling(200).mean())
plt.title('SARSA Learning Curve - Average Revenue per Episode')
plt.xlabel('Episode')
plt.ylabel('Total Reward (Revenue)')
plt.grid(True)
plt.show()

## Part 5: Policy Extraction & Interpretation

**TODO 5.1:** Extract and print the policy as a 3×3 table (rows=time bin, columns=demand level).

In [ ]:
policy = np.argmax(Q, axis=1).reshape(3, 3)

print("Learned Policy (0=Low price, 1=Medium, 2=High price)")
print("Rows: Time remaining (2=Early, 1=Mid, 0=Late)")
print("Columns: Demand (0=Low, 1=Med, 2=High)")
print(policy)

# TODO 5.1: Interpret in words (add print statements or markdown)
# Example: "When demand is high and time is early, the agent chooses action __"

## Part 6: Reflection Questions (Add answers in markdown cells below)

1. How does the learned policy make sense from a data science / business perspective? (e.g., when to discount)
2. Why might SARSA be preferable to Q-Learning in this pricing scenario?
3. If you used the real Kaggle Walmart data, what extra steps would you take to estimate demand sensitivity more accurately?
4. (Optional extension) Download real `train.csv` from Kaggle, filter to one Store+Dept, recompute quantiles and demand effects. Rerun—does policy change?

**Bonus Challenge:** Add IsHoliday as a state feature (expand to more states). How does policy adapt?